# 🌊 OceanGo: AlphaZero Self-Play Training on Google Colab

This notebook trains the **OceanGo** AlphaZero engine utilizing Google Colab's **Cloud TPUs (`v5e-1` with 44 vCPUs)** or **NVIDIA GPUs (`T4`, `L4`, `A100`)**.

### Key Benefits:
- **Massive Multi-Core Self-Play (`TPU v5e-1` or `GPU + High-RAM`)**: Automatically scales across all 44 vCPUs on `v5e-1` (or 8–12 vCPUs on GPU) using direct OS pipe queues and batched neural network inference.
- **Cross-Platform Checkpoints**: Models trained on Colab TPU or GPU save in universal CPU tensor format, ready to play immediately on your Windows PC (NVIDIA GPU) or Mac (Apple Silicon M-series).
- **Direct Google Drive Persistence**: All iteration checkpoints (`policy_value_net_iter_X.pth`) save directly to your Google Drive so nothing is lost upon disconnect.
- **Built-in Dry-Run**: Includes a 10-second verification cell so you never waste compute units on misconfigurations.

## Step 1: Verify Hardware Accelerator (TPU `v5e-1` or GPU `T4`)
In `Runtime -> Change runtime type -> Hardware accelerator`, choose either:
- **TPU `v5e-1`** *(Recommended for maximum CPU self-play speed — provides **44 vCPUs** + TPU bfloat16)*, or
- **T4 GPU** with **High-RAM ON** *(Provides **8 vCPUs** + NVIDIA CUDA Tensor Cores)*.

In [ ]:
import os, subprocess, torch

print(f"PyTorch Version: {torch.__version__}")
print(f"System CPU Cores (vCPUs): {os.cpu_count()}")

if torch.cuda.is_available():
    print(f"Accelerator: NVIDIA CUDA GPU ({torch.cuda.get_device_name(0)})")
    !nvidia-smi
else:
    tpu_env = any(k in os.environ for k in ("TPU_NAME", "COLAB_TPU_ADDR", "PJRT_DEVICE", "XRT_TPU_CONFIG")) or os.path.exists("/dev/accel0") or os.path.exists("/dev/vfio")
    if tpu_env:
        try:
            import torch_xla.core.xla_model as xm
        except ImportError:
            print("Installing torch_xla for Colab TPU v5e-1...")
            subprocess.check_call(["pip", "install", "-q", "torch_xla[tpu]", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"])
            import torch_xla.core.xla_model as xm
        os.environ.setdefault("PJRT_DEVICE", "TPU")
        tpu_dev = xm.xla_device()
        print(f"Accelerator: Google Cloud TPU ({xm.xla_real_devices([str(tpu_dev)])[0]}) with {os.cpu_count()} vCPUs!")
    else:
        print("WARNING: Neither GPU nor TPU detected. Please select TPU v5e-1 or T4 GPU in Runtime -> Change runtime type.")

## Step 2: Mount Google Drive
Mounting Google Drive ensures your checkpoints persist permanently even if Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create dedicated checkpoint directory on your Google Drive
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/OceanGo/checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved directly to: {DRIVE_CHECKPOINT_DIR}")

## Step 3: Clone Repository & Install Dependencies

In [ ]:
# Clone repo (or pull if already present)
import os
if not os.path.isdir('/content/OceanGo'):
    !git clone https://github.com/Ocean48/OceanGo.git /content/OceanGo
else:
    !git -C /content/OceanGo pull

%cd /content/OceanGo
!pip install -r requirements.txt

## Step 4: Dry-Run Verification (Costs ~0 Compute Units)
Runs a quick 10-second end-to-end sanity check to verify CUDA inference, multiprocessing queues, and Google Drive write permissions before starting the full training.

In [ ]:
!python OceanGo.py --dry-run --save-dir /content/drive/MyDrive/OceanGo/checkpoints

## Step 5: Start Full AlphaZero Training
Launches the training pipeline. All models and checkpoints automatically sync to your Google Drive after every iteration, and the runtime **automatically disconnects** when finished so zero Compute Units are wasted on idle time.

### Which Runtime & Preset Should You Choose? (Based on 200 Monthly Compute Units)

| Runtime Setup (`Runtime -> Change runtime type`) | Command Preset | Burn Rate | Time per Loop | 50 Loops Cost & Time | Max Loops per 200 CUs |
| :--- | :--- | :---: | :---: | :---: | :---: |
| 🚀 **TPU `v5e-1`** *(Fastest Self-Play — **44 vCPUs**)* | `--games 200 --sims 400 --iters 50` | **~2.0 CUs/hr** | **~3–5 mins** | **~7–8 CUs (~3.5 hrs)** | **~1,200+ Loops** |
| 🏆 **T4 GPU + High-RAM ON** *(Best GPU Value — 8 vCPUs)* | `--games 200 --sims 400 --iters 50` | **~2.3 CUs/hr** | **~8–10 mins** | **~18 CUs (~8 hrs)** | **~550+ Loops** |
| **TPU `v5e-1`** *(DeepMind Heavy Preset — **44 vCPUs**)* | `--games 500 --sims 800 --iters 50` | **~2.0 CUs/hr** | **~12–16 mins** | **~24 CUs (~12 hrs)** | **~400 Loops** |
| **T4 GPU + High-RAM ON** *(DeepMind Heavy Preset)* | `--games 500 --sims 800 --iters 50` | **~2.3 CUs/hr** | ~35–45 mins | ~75 CUs (~32 hrs) | ~130 Loops |
| **L4 GPU (High-RAM OFF)** *(8 vCPUs, 2x cost)* | `--games 200 --sims 400 --iters 50` | ~4.5 CUs/hr | ~6–8 mins | ~30 CUs (~6.5 hrs) | ~330 Loops |

- **Auto-Resume**: If Colab ever disconnects mid-run, just re-run the notebook—`OceanGo.py` automatically detects your latest `policy_value_net_iter_X.pth` in Google Drive and resumes from `X + 1`.
- **Background Execution**: With Colab Pro / Google AI Pro, you can close this browser tab or shut down your computer while it runs.

In [ ]:
!python OceanGo.py --train \
    --save-dir /content/drive/MyDrive/OceanGo/checkpoints \
    --games 200 \
    --sims 800 \
    --iters 50

# Automatically disconnect and shut down the Colab GPU runtime when training finishes
# so zero Compute Units are wasted on idle time.
from google.colab import runtime
print("Training finished! Automatically disconnecting runtime to save Compute Units...")
runtime.unassign()